In [4]:
import pandas as pd
import numpy as np
from time import perf_counter
from datasets import load_dataset
from memory_profiler import memory_usage
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import BertTokenizer, BertModel


In [5]:
BATCH_SIZE = 32
LEARNING_RATE = 3e-5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRETRAINED_MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
MAX_EPOCHS = 4
PATIENCE = 3

tokenizer = BertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)

print(f"Using device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}, Learning rate: {LEARNING_RATE}")

Using device: cuda
Batch size: 32, Learning rate: 3e-05


In [6]:
ds = load_dataset("cardiffnlp/tweet_eval", "irony")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,text,label
0,seeing ppl walking w/ crutches makes me really...,1
1,"look for the girl with the broken smile, ask h...",0
2,Now I remember why I buy books online @user #s...,1
3,@user @user So is he banded from wearing the c...,1
4,Just found out there are Etch A Sketch apps. ...,1
...,...,...
2857,I don't have to respect your beliefs.||I only ...,0
2858,Women getting hit on by married managers at @u...,1
2859,@user no but i followed you and i saw you post...,0
2860,@user I dont know what it is but I'm in love y...,0


In [7]:
class BinaryClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),

            'labels': torch.tensor([label], dtype=torch.float)
        }


In [8]:
class BertForBinaryClassification(nn.Module):
    def __init__(self):
        super(BertForBinaryClassification, self).__init__()
        self.bert = BertModel.from_pretrained(PRETRAINED_MODEL_NAME)
        self.pre_classifier = nn.Linear(768, 768)
        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Linear(768, 1)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_state = outputs[0][:, 0]
        pooled_output = self.pre_classifier(hidden_state)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        output = self.classifier(pooled_output)
        return torch.sigmoid(output)
    

In [9]:
def get_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precisions, recalls, f1s, supports = precision_recall_fscore_support(y_true, y_pred)

    return acc, precisions, recalls, f1s

In [10]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, save_path, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    start_train = perf_counter()
    
    # Initialize best metrics
    best_train_acc = 0
    best_train_precisions = None
    best_train_recalls = None
    best_train_f1s = None
    best_val_acc = 0
    best_val_precisions = None
    best_val_recalls = None
    best_val_f1s = None
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_true = []
        
        for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{max_epochs}', leave=False):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            train_loss += loss.item()
            preds = (outputs > 0.5).float().cpu().numpy()
            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            loss.backward()
            optimizer.step()
        
        train_loss /= len(train_dataloader)
        train_preds = np.array(train_preds).flatten()
        train_true = np.array(train_true).flatten()
        train_acc, train_precisions, train_recalls, train_f1s = get_metrics(train_true, train_preds)
        
        model.eval()
        val_loss = 0
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in val_dataloader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = (outputs > 0.5).float().cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
        
        val_loss /= len(val_dataloader)
        val_preds = np.array(val_preds).flatten()
        val_true = np.array(val_true).flatten()
        val_acc, val_precisions, val_recalls, val_f1s = get_metrics(val_true, val_preds)
        
        print(f"Epoch {epoch + 1}/{max_epochs} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1s}")
        print(f"Epoch {epoch + 1}/{max_epochs} - Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1s}")
        
        # Early stopping logic
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_train_acc = train_acc
            best_train_precisions = train_precisions
            best_train_recalls = train_recalls
            best_train_f1s = train_f1s
            best_val_acc = val_acc
            best_val_precisions = val_precisions
            best_val_recalls = val_recalls
            best_val_f1s = val_f1s
            torch.save(model.state_dict(), save_path)
            epochs_no_improve = 0
            print("Model saved!")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break
    
    total_train_time = perf_counter() - start_train
    return (best_train_acc, best_train_precisions, best_train_recalls, best_train_f1s,
            best_val_acc, best_val_precisions, best_val_recalls, best_val_f1s, total_train_time)

In [11]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions = []
    true_labels = []
    classification_times = []  # To track time per sample

    start_test = perf_counter()

    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            # Process one sample at a time (for each input in the batch)
            for i in range(input_ids.size(0)):  # Process each sample in the batch
                # Get individual sample
                input_id = input_ids[i].unsqueeze(0)  # Add batch dimension
                attention_mask_sample = attention_mask[i].unsqueeze(0)  # Add batch dimension
                label = labels[i].item()

                # Track the time per sample
                start_time = perf_counter()
                
                # Make prediction
                output = model(input_ids=input_id, attention_mask=attention_mask_sample)
                pred = (output > 0.5).float().cpu().numpy().flatten()[0]
                
                # Append results
                predictions.append(pred)
                true_labels.append(label)

                # Track classification time for each sample
                classification_times.append(perf_counter() - start_time)

    total_test_time = perf_counter() - start_test
    print(f"Test Time: {total_test_time:.2f} seconds")
    
    predictions = np.array(predictions)
    true_labels = np.array(true_labels)

    # Now you can calculate metrics
    acc, precisions, recalls, f1s = get_metrics(true_labels, predictions)

    print("Test Metrics:")
    print("Accuracy:", acc)
    print("F1s:", f1s)
    print("Precisions:", precisions)
    print("Recalls:", recalls)

    return predictions, true_labels


In [12]:
train_texts = train_df['text'].values
train_labels = train_df['label'].values

val_texts = val_df['text'].values
val_labels = val_df['label'].values

test_texts = test_df['text'].values
test_labels = test_df['label'].values

train_dataset = BinaryClassificationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = BinaryClassificationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = BinaryClassificationDataset(test_texts, test_labels, tokenizer, MAX_LEN)

seeds = [2, 3, 5]
results = []

# Grid search loop
for seed in seeds:
    torch.manual_seed(seed)
    model = BertForBinaryClassification().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCELoss()
    
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
    
    save_path = f'results/bert_binary3_bs{BATCH_SIZE}_lr{LEARNING_RATE}_seed{seed}.pt'

    # Train
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    max_memory_usage_train, retval = memory_usage(
        (train_model, (model, train_dataloader, val_dataloader, optimizer, criterion, save_path),
         {'max_epochs': MAX_EPOCHS, 'patience': PATIENCE}), max_usage=True, retval=True)
    
    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    (train_acc, train_precisions, train_recalls, train_f1s,
     val_acc, val_precisions, val_recalls, val_f1s, total_train_time) = retval
    
    # Load best model
    model.load_state_dict(torch.load(save_path))
    
    # Evaluate
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start = perf_counter()
    max_memory_usage_test, test_retval = memory_usage(
        (evaluate_model, (model, test_dataloader), {}), max_usage=True, retval=True)
    total_time_test = perf_counter() - start

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    predictions, true_labels = test_retval
    test_acc, test_precisions, test_recalls, test_f1s = get_metrics(true_labels, predictions)
    
    # Store individual results for this seed
    results.append({
        'seed': seed,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'train_acc': train_acc,
        'train_precisions': train_precisions.tolist(),
        'train_recalls': train_recalls.tolist(),
        'train_f1s': train_f1s.tolist(),
        'max_memory_usage_train': max_memory_usage_train,
        'max_vram_usage_train': max_vram_usage_train,
        'total_train_time': total_train_time,
        'val_acc': val_acc,
        'val_precisions': val_precisions.tolist(),
        'val_recalls': val_recalls.tolist(),
        'val_f1s': val_f1s.tolist(),
        'test_acc': test_acc,
        'test_precisions': test_precisions.tolist(),
        'test_recalls': test_recalls.tolist(),
        'test_f1s': test_f1s.tolist(),
        'max_memory_usage_test': max_memory_usage_test,
        'max_vram_usage_test': max_vram_usage_test,
        'total_test_time': total_time_test
    })

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
Epoch 1/4:   0%|          | 0/90 [00:00<?, ?it/s]c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Epoch 1/4 - Train Loss: 0.6335, Acc: 0.6212, F1: [0.59612519 0.64342105]
Epoch 1/4 - Val Loss: 0.5867, Acc: 0.6942, F1: [0.73928571 0.63037975]
Model saved!


Epoch 2/4 - Train Loss: 0.4829, Acc: 0.7806, F1: [0.78011204 0.78103208]
Epoch 2/4 - Val Loss: 0.6045, Acc: 0.7058, F1: [0.71701913 0.69356598]


Epoch 3/4 - Train Loss: 0.2798, Acc: 0.8889, F1: [0.88896648 0.88881119]
Epoch 3/4 - Val Loss: 0.7628, Acc: 0.7026, F1: [0.69593148 0.70901639]


Epoch 4/4 - Train Loss: 0.1088, Acc: 0.9626, F1: [0.9622841  0.96293731]
Epoch 4/4 - Val Loss: 0.9181, Acc: 0.7068, F1: [0.73733583 0.66824645]
Early stopping triggered


...

Epoch 2/4 - Train Loss: 0.5191, Acc: 0.7537, F1: [0.74991132 0.75731497]
Epoch 2/4 - Val Loss: 0.5731, Acc: 0.6974, F1: [0.70358974 0.69090909]
Model saved!


Epoch 3/4 - Train Loss: 0.3099, Acc: 0.8732, F1: [0.87159533 0.87469796]
Epoch 3/4 - Val Loss: 0.7452, Acc: 0.6785, F1: [0.64260768 0.70789724]


Epoch 4/4 - Train Loss: 0.1105, Acc: 0.9633, F1: [0.96306718 0.96355432]
Epoch 4/4 - Val Loss: 1.0474, Acc: 0.6524, F1: [0.60570071 0.68913858]


C:\Users\Rafael\AppData\Local\Temp\ipykernel_15084\4271812077.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(save_path))
c:\Users\Rafa

Test Time: 6.52 seconds
Test Metrics:
Accuracy: 0.6900510204081632
F1s: [0.72788354 0.64      ]
Precisions: [0.77380952 0.59340659]
Recalls: [0.68710359 0.69453376]


In [13]:
df = pd.DataFrame(results)
df.to_csv('results/bert_binary3.csv', index=False)

In [14]:
df

,seed,batch_size,learning_rate,train_acc,train_precisions,train_recalls,train_f1s,max_memory_usage_train,max_vram_usage_train,total_train_time,...,val_precisions,val_recalls,val_f1s,test_acc,test_precisions,test_recalls,test_f1s,max_memory_usage_test,max_vram_usage_test,total_test_time
0,2,32,0.00003,0.621244,"[0.6314127861089187, 0.613166144200627]","[0.5645730416372619, 0.6768166089965398]","[0.5961251862891207, 0.6434210526315789]",1316.097656,3806.391113,122.899765,...,"[0.6666666666666666, 0.7455089820359282]","[0.8296593186372746, 0.5460526315789473]","[0.7392857142857143, 0.6303797468354431]",0.702806,"[0.7298850574712644, 0.648854961832061]","[0.8054968287526427, 0.5466237942122186]","[0.7658291457286432, 0.5933682373472949]",1242.988281,1759.537109,7.512541
1,3,32,0.00003,0.761006,"[0.7515442690459849, 0.7708185053380783]","[0.7727593507410021, 0.7494809688581315]","[0.7620041753653445, 0.76]",1323.824219,3826.016113,123.498764,...,"[0.708803611738149, 0.638671875]","[0.6292585170340681, 0.7171052631578947]","[0.6666666666666666, 0.6756198347107438]",0.677296,"[0.7657004830917874, 0.5783783783783784]","[0.6701902748414377, 0.6881028938906752]","[0.7147688838782412, 0.6284875183553598]",1245.242188,1769.162109,7.122967
2,5,32,0.00003,0.753669,"[0.753922967189729, 0.7534246575342466]","[0.7459421312632322, 0.7612456747404844]","[0.7499113160695282, 0.7573149741824441]",1324.367188,3816.391113,113.924575,...,"[0.7205882352941176, 0.6743215031315241]","[0.687374749498998, 0.7083333333333334]","[0.7035897435897436, 0.6909090909090909]",0.690051,"[0.7738095238095238, 0.5934065934065934]","[0.6871035940803383, 0.6945337620578779]","[0.7278835386338186, 0.64]",1245.722656,1769.912109,7.017236
